In [1]:
import os
import numpy as np
import pandas as pd
import boto3
import time
import helper
import sys
cwd = os.getcwd()
print(cwd)


/home/sagemaker-user/CAPE_PERFORMANCE


#### Import Cape data

Load the May and July data.

In [2]:
# read data from May
bucket_name = 'pr-home-datascience'
prefix = 'Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/'

may_keywords = 'processed_combinedv4v5.csv'
may_data_all = helper.read_s3(bucket_name, prefix, may_keywords)

july_v4_keywords = 'v4.csv.gz'
july_v4 =  helper.read_s3(bucket_name, prefix, july_v4_keywords)

july_v5_keywords = 'v5.csv.gz'
july_v5 =  helper.read_s3(bucket_name, prefix, july_v5_keywords)

Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/PRH_UWLookback_processed_combinedv4v5.csv
Load file successfully, file length is  757655
Now the total rows are  757655
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/PRAC_Historical_PIF_072025_Updated_073125_cape_processed_rcrv4.csv.gz
Load file successfully, file length is  2000358
Now the total rows are  2000358
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/PRAC_Historical_PIF_072025_Updated_073125_cape_processed_rcrv5.csv.gz
Load file successfully, file length is  2000358
Now the total rows are  2000358


Split the May data into V4 and V5.

In [3]:
# split the May dataset into V4 and V5, keep the variables not begining with v4_ or v5_
cols = may_data_all.columns

# Separate the columns
v4_cols = [c for c in cols if c.startswith("v4_")]
v5_cols = [c for c in cols if c.startswith("v5_")]
shared_cols = [c for c in cols if not (c.startswith("v4_") or c.startswith("v5_"))]

# Build separate DataFrames
may_v4 = may_data_all[shared_cols + v4_cols].copy()
may_v5 = may_data_all[shared_cols + v5_cols].copy()

# (Optional) rename v4_ / v5_ columns to remove the prefix for easier comparison
may_v4 = may_v4.rename(columns=lambda x: x.replace("v4_", "") if x.startswith("v4_") else x)
may_v5 = may_v5.rename(columns=lambda x: x.replace("v5_", "") if x.startswith("v5_") else x)
may_v4["cape_run_dt"] = may_v4["effective_date"]
may_v5["cape_run_dt"] = may_v5["effective_date"]

Check the variables

In [4]:
keywords = 'pol_num'
key_columns = [col for col in may_v4.columns if keywords in col.lower()]
print('There are ', len(key_columns), ' variables contains ' + keywords + '.')
print("\n".join(key_columns))

There are  1  variables contains pol_num.
pol_num


#### Import the model data

In [7]:

# import all the data from Model
bucket_name = 'pr-home-datascience'
prefix = 'DSwarehouse/Datasources/PolicyClaims/BHclaim_dvs_add/rundt=202510/'
s3 = boto3.client('s3')

response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

if 'Contents' in response:
    parquet_files = [obj['Key'] for obj in response['Contents'] if obj['Key'].endswith('.parquet')]
    print("Found parquet files:")
    for f in parquet_files:
        print(f)
else:
    print("No files found under that prefix.")


states = ["CT", "MA", "NH", "NJ", "NY", "PA"]
# states = ["CT", "MA"]

dfs = []

for s in states:
    path = f"s3://{bucket_name}/DSwarehouse/Datasources/PolicyClaims/BHclaim_dvs_add/rundt=202510/state={s}/0e848644c6284d54ae247f4c6a406145-0.parquet"
    print(f"Reading {s}...")
    df = pd.read_parquet(path)
    df["state"] = s
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print("Merged total rows:", len(df_all))


Found parquet files:
DSwarehouse/Datasources/PolicyClaims/BHclaim_dvs_add/rundt=202510/state=CT/0e848644c6284d54ae247f4c6a406145-0.parquet
DSwarehouse/Datasources/PolicyClaims/BHclaim_dvs_add/rundt=202510/state=MA/0e848644c6284d54ae247f4c6a406145-0.parquet
DSwarehouse/Datasources/PolicyClaims/BHclaim_dvs_add/rundt=202510/state=NH/0e848644c6284d54ae247f4c6a406145-0.parquet
DSwarehouse/Datasources/PolicyClaims/BHclaim_dvs_add/rundt=202510/state=NJ/0e848644c6284d54ae247f4c6a406145-0.parquet
DSwarehouse/Datasources/PolicyClaims/BHclaim_dvs_add/rundt=202510/state=NY/0e848644c6284d54ae247f4c6a406145-0.parquet
DSwarehouse/Datasources/PolicyClaims/BHclaim_dvs_add/rundt=202510/state=PA/0e848644c6284d54ae247f4c6a406145-0.parquet
Reading CT...
Reading MA...
Reading NH...
Reading NJ...
Reading NY...
Reading PA...
Merged total rows: 4330999


In [8]:
keywords = ['co_cd', 'cur_term_eff_dt', 'qpid', 'ncat', 'ncat_cnt', 'ee', 'eprm', 'new_or_rnwl_flg',
            'zip', 'zip4', 'state', 'year', 'tv', 'pol_num', 'form']

model_data = df_all[ keywords]

print(len(model_data))

model_data = model_data[model_data['form'] == 'HO_3']
print(model_data['form'].unique())
print(len(model_data))


model_data = model_data[model_data['co_cd'] != 'HPPREF']
print(len(model_data))


4330999
<StringArray>
['HO_3']
Length: 1, dtype: string
3418358
1864631


#### Matching model and Cape

The matched process is the following: For May data, use the [‘pol_num’, ‘cur_term_eff_dt’] in model to match with the [‘pol_num’, ‘effective_date’] in cape. The matched ones should have the same pol_num, the effective_date should be no later than than (<=) cur_term_eff_dt. For July data, use the ‘cape_run_dt’ instead in Cape and the cape_run_dt should be early than (<) cur_term_eff_dt in model.

In [9]:
def match_cape_prior_fast(
    model: pd.DataFrame,
    cape: pd.DataFrame,
    key: str = "pol_num",
    model_date_col: str = "cur_term_eff_dt",
    cape_date_col: str = "effective_date",
    allow_exact_matches: bool = True,
):
    """
    Return two aligned DataFrames of matched samples only.
    For each model row (same key), pick the most recent cape date <= model_date.
    If a key has no prior cape date for a given model row, that model row is skipped.

    Output:
      model_matched: rows from `model`
      cape_matched : corresponding rows from `cape` (same length, row-aligned)
    """

    # --- Normalize key & datetimes; ensure key is a column ---
    mdl = model.copy()
    cap = cape.copy()

    if key not in mdl.columns and mdl.index.name == key:
        mdl = mdl.reset_index()
    if key not in cap.columns and cap.index.name == key:
        cap = cap.reset_index()

    mdl[key] = mdl[key].astype(str).str.strip().str.upper()
    cap[key] = cap[key].astype(str).str.strip().str.upper()
    mdl[model_date_col] = pd.to_datetime(mdl[model_date_col], errors="coerce", utc=False)
    cap[cape_date_col]  = pd.to_datetime(cap[cape_date_col],  errors="coerce", utc=False)
    mdl = mdl.dropna(subset=[model_date_col])
    cap = cap.dropna(subset=[cape_date_col])

    if mdl.empty or cap.empty:
        return mdl.iloc[0:0].copy(), cap.iloc[0:0].copy()

    # --- Sort by (key, date) and remember original positions ---
    mdl_sorted = mdl.sort_values([key, model_date_col], kind="mergesort").copy()
    cap_sorted = cap.sort_values([key, cape_date_col], kind="mergesort").copy()
    mdl_sorted["_orig_idx_m"] = mdl_sorted.index.to_numpy()
    cap_sorted["_orig_idx_c"] = cap_sorted.index.to_numpy()

    # --- Build quick access structures for cape per key ---
    cap_groups = cap_sorted.groupby(key, sort=False)
    # Map key -> slice (start,end) in cap_sorted to avoid copying arrays repeatedly
    cap_pos_map = {}
    start = 0
    for k_val, g in cap_groups:
        end = start + len(g)
        cap_pos_map[k_val] = (start, end)
        start = end

    cap_dates_all = cap_sorted[cape_date_col].to_numpy()
    cap_idx_all   = cap_sorted["_orig_idx_c"].to_numpy()

    # --- Iterate model groups and pick prior cape with searchsorted ---
    mdl_out_idx = []
    cap_out_idx = []

    for k_val, g_m in mdl_sorted.groupby(key, sort=False):
        if k_val not in cap_pos_map:
            continue  # skip keys not in cape

        s, e = cap_pos_map[k_val]
        c_dates = cap_dates_all[s:e]     # sorted cape dates for this key
        c_orig  = cap_idx_all[s:e]       # original cape indices for this key

        m_dates = g_m[model_date_col].to_numpy()
        # index of rightmost cape date <= m_date
        idx = np.searchsorted(c_dates, m_dates, side="right") - 1
        valid = idx >= 0

        if not allow_exact_matches:
            # remove exact equals (strict prior only)
            valid_idx = np.flatnonzero(valid)  # positions where valid==True
            eq_compact = (c_dates[idx[valid]] == m_dates[valid])  # length = valid.sum()
            # write back into the full-length mask
            valid[valid_idx[eq_compact]] = False

        if not valid.any():
            continue

        # Collect original row indices for matched pairs
        mdl_out_idx.extend(g_m.loc[valid, "_orig_idx_m"].to_list())
        cap_out_idx.extend(c_orig[idx[valid]])

    if not mdl_out_idx:
        return mdl.iloc[0:0].copy(), cap.iloc[0:0].copy()

    # --- Slice original dataframes and align order ---
    model_matched = mdl.loc[mdl_out_idx].reset_index(drop=True)
    cape_matched  = cap.loc[cap_out_idx].reset_index(drop=True)

    return model_matched, cape_matched

Note: please change the cape_date_col and allow_exact_matches accordingly

In [21]:

# for July the cape_run_dt should be strictly earlier than the cur_term_eff_dt
model_matched, cape_matched = match_cape_prior_fast(
    model=model_data,
    cape=may_v4,
    key="pol_num",
    model_date_col="cur_term_eff_dt",
    cape_date_col="effective_date", # effective_date or cape_run_dt
    allow_exact_matches=True,  # set False if you need strictly earlier # July is False, May is True
)




Show the results not matching and check.

In [26]:
def pol_nums_not_matched(model, cape, *, key="pol_num",
                         model_date_col="cur_term_eff_dt", cape_date_col="effective_date",
                         allow_exact_matches=True):
    # use your matcher
    model_matched, _ = match_cape_prior_fast(
        model, cape, key=key,
        model_date_col=model_date_col, cape_date_col=cape_date_col,
        allow_exact_matches=allow_exact_matches,
    )
    # normalize like the matcher
    mdl = model.copy()
    if key not in mdl.columns and mdl.index.name == key:
        mdl = mdl.reset_index()
    all_keys = mdl[key].astype(str).str.strip().str.upper()
    matched_keys = model_matched[key].astype(str).str.strip().str.upper()

    # keys that never appeared in the matched set
    unmatched_keys = pd.Index(all_keys).difference(pd.Index(matched_keys))
    # return as a sorted Series (one row per unique pol_num)
    return pd.Series(sorted(unmatched_keys.unique()), name=key)



In [34]:

not_matched_keys_may = pol_nums_not_matched(model_data, may_v4,model_date_col="cur_term_eff_dt", cape_date_col="effective_date",allow_exact_matches=True)
not_matched_keys_july = pol_nums_not_matched(model_data, july_v4,model_date_col="cur_term_eff_dt", cape_date_col="cape_run_dt",allow_exact_matches=False)

may_set = set(map(str, not_matched_keys_may.tolist()))
jul_set = set(map(str, not_matched_keys_july.tolist()))

both_unmatched = sorted(may_set & jul_set)  # in BOTH not-matched lists

print(f"{len(both_unmatched)} pol_num not matched by BOTH May and July:")
print(", ".join(both_unmatched[:50]), "..." if len(both_unmatched) > 50 else "")

# as a Series / DataFrame if you want to save or view
both_unmatched_ser = pd.Series(both_unmatched, name="pol_num")

90132 pol_num not matched by BOTH May and July:
BHD00001001021, BHD00001001025, BHD00001001026, BHD00001001027, BHD00001001028, BHD00001001029, BHD00001001031, BHD00001001032, BHD00001001033, BHD00001001034, BHD00001001035, BHD00001001036, BHD00001001038, BHD00001001040, BHD00001001043, BHD00001001044, BHD00001001045, BHD00001001046, BHD00001001047, BHD00001001051, BHD00001001052, BHD00001001055, BHD00001001056, BHD00001001058, BHD00001001060, BHD00001001063, BHD00001001069, BHD00001001070, BHD00001001072, BHD00001001073, BHD00001001080, BHD00001001082, BHD00001001084, BHD00001001085, BHD00001001086, BHD00001001087, BHD00001001089, BHD00001001090, BHD00001001092, BHD00001001093, BHD00001001094, BHD00001001095, BHD00001001096, BHD00001001097, BHD00001001099, BHD00001001100, BHD00001001101, BHD00001001102, BHD00001001103, BHD00001001105 ...


In [38]:

# print(model_data.loc[model_data['pol_num'] == 'BHD00001001021'])
print(may_v4.loc[may_v4['pol_num'] == 'BHD00001001021'])
print(july_v4.loc[july_v4['pol_num'] == 'BHD00001001021'])

Empty DataFrame
Columns: [source row number, cape_response_status, cape_response_description, cape_response_id, cape_primary_structure_latitude, cape_primary_structure_longitude, attribute_geometry_id, cape_parcel_id, cape_oblique_property_date, cape_oblique_property_image_source, cape_oblique_property_image_url, cape_parcel_imagery_date, cape_parcel_imagery_image_source, cape_parcel_imagery_image_url, address1, city, county, effective_date, house_number, orgl_pol_eff_dt, pol_num, property_address, row_id, state, street_name, street_type, zipcode, cape_geocode_confidence, cape_geocode_method, cape_review_url, cape_property_condition_report_url, cape_accessory_structure_count, cape_accessory_structure_count_date, cape_accessory_structure_count_image_source, cape_accessory_structure_count_image_url, cape_accessory_structure_footprint, cape_accessory_structure_roof_condition_rating, cape_accessory_structure_roof_condition_rating_confidence, cape_accessory_structure_roof_condition_rating_d

In [19]:
model_matched.head(10)
# keywords = 'cur_eff'

# key_columns = [col for col in cape_matched.columns if keywords in col.lower()]
# print('There are ', len(key_columns), ' variables contains ' + keywords + '.')
# print("\n".join(key_columns))

,co_cd,cur_term_eff_dt,qpid,ncat,ncat_cnt,ee,eprm,new_or_rnwl_flg,zip,zip4,state,year,tv,pol_num
0,BHIC,2014-01-03,65164516,0.0,0,0.999993,1353.000001,1,2740,1228,MA,2014,T,BHD00001001024
1,BHIC,2015-01-03,65164516,0.0,0,0.999993,1377.000008,1,2740,1228,MA,2015,T,BHD00001001024
2,BHIC,2016-01-03,65164516,0.0,0,0.999994,2211.999982,1,2740,1228,MA,2016,T,BHD00001001024
3,BHIC,2017-01-03,65164516,0.0,0,0.999993,1994.999994,1,2740,1228,MA,2017,T,BHD00001001024
4,BHIC,2018-01-03,65164516,0.0,0,0.999993,2057.999993,1,2740,1228,MA,2018,T,BHD00001001024
5,BHIC,2019-01-03,65164516,0.0,0,0.999993,2127.999992,1,2740,1228,MA,2019,T,BHD00001001024
6,BHIC,2020-01-03,65164516,0.0,0,0.999994,2208.999983,1,2740,1228,MA,2020,T,BHD00001001024
7,BHIC,2021-01-03,65164516,0.0,0,0.865748,1997.999992,1,2740,1228,MA,2021,T,BHD00001001024
8,BHIC,2014-01-27,65173580,0.0,0,0.999992,1203.000007,1,2740,4523,MA,2014,V,BHD00001001030
9,BHIC,2015-01-27,65173580,0.0,0,0.999992,1479.999995,1,2740,4523,MA,2015,V,BHD00001001030


Save the results for model and Cape. We don't merge them together so that we can examine them later.

In [22]:
output_path = 's3://pr-home-datascience/Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/'


# Write to Excel safely, disabling hyperlink auto-detection
model_matched.to_csv(output_path + 'model_may_v4.csv', index=False)
cape_matched.to_csv(output_path + 'cape_may_v4.csv', index=False)

print(f"✅ Saved matched results to {output_path}")
print(f"   Total matched pairs: {len(model_matched)}")

✅ Saved matched results to s3://pr-home-datascience/Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/
   Total matched pairs: 616091


Load the data and examine the results distribution.

In [11]:
bucket_name = 'pr-home-datascience'
prefix = 'Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/'

may_keywords = 'cape_july_v4.csv'
may_data_v4 = helper.read_s3(bucket_name, prefix, may_keywords)

model_keywords = 'model_july_v4.csv'
model_data_v4 = helper.read_s3(bucket_name, prefix, model_keywords)



Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/cape_july_v4.csv
Load file successfully, file length is  725172
Now the total rows are  725172
Reading from the file  Projects/AdHoc/InternProjects/2025/2025FallCapeV5_vs_V4_performance_analysis/data_aligned/model_july_v4.csv
Load file successfully, file length is  725172
Now the total rows are  725172


In [12]:
print(f'Cape July data V,4 length of {len(may_data_v4)}')
print(may_data_v4[['pol_num', 'cape_run_dt']][0:20])
print(f'Model corresponding data, length of {len(model_data_v4)}')
print(model_data_v4[['pol_num', 'cur_term_eff_dt']][0:20])

Cape July data V,4 length of 725172
           pol_num cape_run_dt
0   BHD00001001024  2013-12-03
1   BHD00001001024  2014-12-03
2   BHD00001001024  2015-12-03
3   BHD00001001024  2016-12-03
4   BHD00001001024  2017-12-03
5   BHD00001001024  2017-12-03
6   BHD00001001024  2017-12-03
7   BHD00001001024  2017-12-03
8   BHD00001001030  2013-12-27
9   BHD00001001030  2014-12-27
10  BHD00001001030  2015-12-27
11  BHD00001001030  2016-12-27
12  BHD00001001030  2017-12-27
13  BHD00001001030  2017-12-27
14  BHD00001001030  2017-12-27
15  BHD00001001030  2017-12-27
16  BHD00001001030  2017-12-27
17  BHD00001001030  2017-12-27
18  BHD00001001030  2017-12-27
19  BHD00001001037  2014-01-06
Model corresponding data, length of 725172
           pol_num cur_term_eff_dt
0   BHD00001001024      2014-01-03
1   BHD00001001024      2015-01-03
2   BHD00001001024      2016-01-03
3   BHD00001001024      2017-01-03
4   BHD00001001024      2018-01-03
5   BHD00001001024      2019-01-03
6   BHD00001001024      2

In [13]:
print(f'Cape May data V4, length of {len(may_data_v4)}')
print(may_data_v4[['pol_num', 'effective_date']][0:20])
print(f'Model corresponding data, length of {len(model_data_v4)}')
print(model_data_v4[['pol_num', 'cur_term_eff_dt']][0:20])

Cape May data V4, length of 725172


KeyError: "['effective_date'] not in index"